# Steering & composition tests

Loads the control-vector bundle produced by `extract_steering_vectors.ipynb`, applies vectors to a
middle layer band with repeng's `ControlModel`, and runs:

1. **Single-mechanism coefficient sweep** — does the vector induce the mechanism? at what strength
   does coherence break?
2. **Syndrome composition** — do summed atom vectors (`personas.SYNDROME_RECIPES`) reproduce the
   disorder?
3. **Compound emergence** — do catastrophizing / learned-helplessness emerge from their predicted
   atoms (`personas.COMPOUND_PREDICTIONS`)?

No vLLM here — steering needs the HF model directly. Needs a GPU.

## 1. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ChuloIva/Predictive_coding"

# Locate steering_lab: running locally inside it, sitting at the repo root, or a fresh Colab clone.
def _find_lab():
    here = Path.cwd()
    if (here / "steering" / "basins.py").exists():
        return here                          # already inside steering_lab
    if (here / "steering_lab" / "steering" / "basins.py").exists():
        return here / "steering_lab"         # at the repo root
    return None

LAB_DIR = _find_lab()
if LAB_DIR is None:
    repo = Path.cwd() / "Predictive_coding"
    if not repo.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo)])
    LAB_DIR = repo / "steering_lab"
os.chdir(LAB_DIR)
sys.path.insert(0, str(LAB_DIR))
print("steering_lab:", os.getcwd())


In [ ]:
# Full precision (bf16) on a large-GPU Colab runtime — no quantization. Gemma 4 needs latest transformers.
%pip install -q -U transformers accelerate torch
%pip install -q repeng sentencepiece matplotlib openai
# repeng = steering · sentencepiece = Gemma tokenizer · matplotlib = basin plots · openai = optional judge


In [ ]:
# Keys: HF_TOKEN (optional — Gemma 4 is Apache-2.0/ungated) + OPENROUTER_API_KEY (optional judge).
import os
from pathlib import Path
_KEYS = ["HF_TOKEN", "OPENROUTER_API_KEY"]
try:
    from google.colab import userdata  # type: ignore
    for k in _KEYS:
        if not os.environ.get(k):
            try:
                v = userdata.get(k)
                if v: os.environ[k] = v
            except Exception: pass
except ImportError:
    pass
try:
    from dotenv import load_dotenv
    for p in [LAB_DIR / ".env", Path.cwd() / ".env"]:
        if p.exists(): load_dotenv(p, override=False)
except ImportError:
    pass
for k in _KEYS:
    print(f"{k:20s}: {'set' if os.environ.get(k) else 'not set'}")


## 2. Load the control-vector bundle

In [ ]:
from pathlib import Path
from steering import extract

# Look on Drive first, then the local extraction output, else prompt for upload.
_candidates = [
    Path("/content/drive/MyDrive/steering_pilot/control_vectors.pkl"),
    LAB_DIR / "steering" / "out" / "control_vectors.pkl",
]
BUNDLE_PATH = next((str(p) for p in _candidates if p.exists()), None)
if BUNDLE_PATH is None:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        if _candidates[0].exists(): BUNDLE_PATH = str(_candidates[0])
    except Exception:
        pass
if BUNDLE_PATH is None:
    from google.colab import files  # type: ignore
    up = files.upload()                       # pick control_vectors.pkl
    BUNDLE_PATH = next(iter(up))

bundle = extract.load_bundle(BUNDLE_PATH)
vecs = extract.bundle_to_control_vectors(bundle)   # {mechanism: ControlVector}
print("loaded", len(vecs), "vectors from", BUNDLE_PATH)
print("mechanisms:", list(vecs))
print("model:", bundle["model_name"], "| method:", bundle["method"])


## 3. Select & load the model (wrap in ControlModel)

Pick any HF causal / multimodal LM with `MODEL_NAME` — default `google/gemma-4-12B-it` (from
`config.py`); override by editing the cell or setting the `STEER_MODEL` env var. Loaded **full
precision (bf16)** with `device_map="auto"`, so use a large-GPU Colab runtime (A100 / L4). Gemma 4
is encoder-free multimodal but runs text-only here and steers like any causal LM. **Control vectors
are model-specific:** the loaded bundle must have been extracted on the *same* `MODEL_NAME`, or the
steering cells won't be meaningful (the cell warns on a mismatch).

In [ ]:
import os
from steering import steer, config

# --- model selection (edit MODEL_NAME, or set the STEER_MODEL env var) ---
MODEL_NAME = os.environ.get("STEER_MODEL", config.MODEL_NAME)   # default: google/gemma-4-12B-it
print(f"loading {MODEL_NAME}  (full precision bf16, device_map=auto) ...")

model, tok = steer.load_model_and_tokenizer(
    MODEL_NAME, dtype="bfloat16", device_map="auto", hf_token=os.environ.get("HF_TOKEN"))

# control vectors are model-specific — warn if the loaded bundle came from a different model
if "bundle" in globals() and bundle.get("model_name") and bundle["model_name"] != MODEL_NAME:
    print(f"  WARNING: bundle was extracted on {bundle['model_name']!r}, not {MODEL_NAME!r} —")
    print("           control vectors will NOT transfer; re-extract on this model first.")

n = len(steer.decoder_layers(model))           # robust for multimodal wrappers (vs config.num_hidden_layers)
BAND = list(range(n // 3, 2 * n // 3))          # middle third
cmodel = steer.make_control_model(model, BAND)   # NOTE: mutates `model`
print(f"{n} layers; steering band = {BAND[0]}..{BAND[-1]}")


## 4. Single-mechanism coefficient sweep

`coeff=0` is the clean baseline. Increase until the mechanism shows; back off when coherence
breaks. If output turns to gibberish/repetition at high coeff, set `normalize=True` (rescales to
the original activation magnitude) or lower the coeff.

In [ ]:
from steering import steer

PROBES = [
    "Tell me about your day.",
    "What are your plans for the weekend?",
    "How is work going lately?",
]
MECH   = "rumination"            # <- change to any mechanism in `vecs`
COEFFS = [0.0, 2.0, 4.0, 8.0]    # <- tune to your model/band
vec = vecs[MECH]

prompt = PROBES[0]
print("PROMPT:", prompt)
for c in COEFFS:
    txt = steer.generate_steered(cmodel, tok, prompt, vec, c,
                                 normalize=False, max_new_tokens=150)
    print(f"\n--- {MECH} @ coeff={c} ---\n{txt}")

In [ ]:
# Optional LLM judge: presence (does the mechanism show?) + coherence, 0-10, across the sweep.
import os
from steering import steer, personas
if not os.environ.get("OPENROUTER_API_KEY"):
    print("No OPENROUTER_API_KEY — skipping judge (eyeball the outputs above instead).")
else:
    desc = personas.PERSONA_BY_ID[MECH].on
    print(f"{MECH}\n{'coeff':>7}{'presence':>10}{'coherence':>11}")
    for c in COEFFS:
        txt = steer.generate_steered(cmodel, tok, PROBES[0], vec, c, max_new_tokens=150)
        s = steer.judge_text(txt, desc)
        print(f"{c:>7}{str(s['presence']):>10}{str(s['coherence']):>11}")

## 5. Syndrome composition

Composed vector = sum of atom vectors. Because magnitudes stack with the number of atoms, use a
smaller per-syndrome coeff (or weight atoms by `1/len`). Does the disorder signature appear?

In [ ]:
from steering import steer, personas

SYN_COEFF = 3.0      # lower than single-mechanism: summed atoms stack in magnitude
prompt = PROBES[0]
print("PROMPT:", prompt, "\n")
# baseline for reference
print("--- BASELINE (no steering) ---\n" +
      steer.generate_steered(cmodel, tok, prompt, None, 0.0, max_new_tokens=160) + "\n")
for syn, atoms in personas.SYNDROME_RECIPES.items():
    cvec = steer.compose(vecs, atoms)           # equal-weight sum; or coeffs=[...]
    txt = steer.generate_steered(cmodel, tok, prompt, cvec, SYN_COEFF, max_new_tokens=160)
    print("="*80)
    print(f"{syn.upper()}  =  {' + '.join(atoms)}\n\n{txt}\n")

## 6. Compound emergence (falsifiable check)

If catastrophizing / learned-helplessness emerge from their predicted atoms, the atom/compound
split in `mechanism_syndrome_map.md` holds. If not, that compound may be its own atom.

In [ ]:
from steering import steer, personas

prompt = PROBES[0]
for compound, atoms in personas.COMPOUND_PREDICTIONS.items():
    cvec = steer.compose(vecs, atoms)
    txt = steer.generate_steered(cmodel, tok, prompt, cvec, SYN_COEFF, max_new_tokens=160)
    print("="*80)
    print(f"{compound.upper()}  =  {' + '.join(atoms)}\n\n{txt}\n")

## Notes

- Sweep `BAND`, `COEFFS`, and `normalize` per mechanism — the usable window differs by vector.
- For a cleaner read, run each cell over all of `PROBES`, not just `PROBES[0]`.
- Separability recap: `extract.cosine_matrix(vecs, layer)` shows which atoms are near-orthogonal
  (compose cleanly) vs entangled (interfere).
- `compose(vecs, atoms, coeffs=[...])` lets you weight atoms (e.g., the RDoC-style mixtures).

## 7. Experiment ③ — metastability probe (trimmed robust core)

Tests one falsifiable sentence: **steering toward a looping pathology reduces residual-stream
trajectory flexibility and warps the internal representation — reversibly (REBUS = −coeff), and
distinguishably from a norm-matched random steer.** Friston's "loss of metastability" is the
motivation, not a claim the numbers carry. Locked spec + predictions: `context/metastability_prereg.md`.

Four conditions (matched magnitude): `clean` · `looping` · `REBUS` · `random` (keystone control).
Two stimulus families: content-free prompts (the model *talks* → trajectory) and a fixed everyday
read-only battery (the model *reads* → surprisal/entropy on aligned tokens). Needs the loaded model
+ `vecs` from the cells above; optionally a PC bundle for the REBUS `prior_precision_high` vector.

In [ ]:
# Experiment ③ — trimmed metastability probe. See context/metastability_prereg.md for the locked spec.
from pathlib import Path
from steering import experiment, extract
from steering.metastability import SICK_DIRECTION  # noqa: F401  (direction reference)

# --- pick the looping pathology vector from the clinical bundle ---
LOOP = "rumination"                 # or "circular_inference" if you also load the PC bundle into `vecs`
looping_vec = vecs[LOOP]

# --- REBUS wants the PC over-precise-prior vector; fall back to a coarse −coeff on the looping vector ---
rebus_vec = None
for _p in [Path("/content/drive/MyDrive/steering_pilot/control_vectors_pc.pkl"),
           LAB_DIR / "steering" / "out" / "control_vectors_pc.pkl"]:
    if _p.exists():
        _pc = extract.bundle_to_control_vectors(extract.load_bundle(str(_p)))
        rebus_vec = _pc.get("prior_precision_high")
        print("REBUS uses PC 'prior_precision_high' from", _p)
        break
if rebus_vec is None:
    print("No PC bundle found — REBUS falls back to −coeff on the looping vector (coarse; documented).")

# --- read at a layer just past the top of the steered band, to capture the accumulated steer ---
READ_LAYER = min(BAND[-1] + 1, n)   # index into out.hidden_states (0 = embeddings, i = output of layer i-1)
COEFF = 8.0
print(f"looping={LOOP}  read_layer={READ_LAYER}  coeff={COEFF}  band={BAND[0]}..{BAND[-1]}")

res = experiment.run_experiment(
    cmodel, tok,
    looping_vec=looping_vec, rebus_vec=rebus_vec,
    read_layer=READ_LAYER, coeff=COEFF, seed=0,
    gen={"max_new_tokens": 160},
)

# --- per-condition tables ---
CONDS = ["clean", "looping", "REBUS", "random"]
def _show(title, agg, keys):
    print(f"\n=== {title} ===")
    print(f"{'metric':22s}" + "".join(f"{c:>12s}" for c in CONDS))
    for k in keys:
        print(f"{k:22s}" + "".join(f"{agg.get(c, {}).get(k, float('nan')):>12.3f}" for c in CONDS))

_show("generative (trajectory / drift / cross-surprise / repetition)", res["agg_generative"],
      ["participation_ratio", "lag1_autocorr", "mean_step",
       "drift_from_prior", "cross_surprise", "distinct_2", "token_entropy"])
_show("read-only (precision / accuracy / drift)", res["agg_readonly"],
      ["next_token_entropy", "surprisal", "drift_from_prior"])

# --- headline prediction checks ---
g, r = res["agg_generative"], res["agg_readonly"]
checks = {
    "looping PR ↓ vs clean":                 g["looping"]["participation_ratio"] < g["clean"]["participation_ratio"],
    "looping autocorr ↑ vs clean":           g["looping"]["lag1_autocorr"] > g["clean"]["lag1_autocorr"],
    "looping entropy ↓ vs clean (read-only)": r["looping"]["next_token_entropy"] < r["clean"]["next_token_entropy"],
    "REBUS restores PR (↑ vs looping)":      g["REBUS"]["participation_ratio"] > g["looping"]["participation_ratio"],
    "random ≠ looping (cross_surprise ↑)":   g["random"]["cross_surprise"] > g["looping"]["cross_surprise"],
}
print("\nprediction checks:")
for k, v in checks.items():
    print(f"  {'PASS' if v else 'fail'}  {k}")


## 8. Experiment ③·b — basin landscape (drawing the metastability claim)

Where Experiment ③ measures trajectory flexibility as *numbers*, this draws the residual-stream
basin as a *picture*: the grid-teleport flow-field method of Fernando & Guitchounts (arXiv 2502.12131),
reimplemented in `steering/basins.py` and retargeted to this model — but with the steer **active
during injection**, so the same grid is teleported under `clean` / `looping` / `REBUS` / `random`.
Arrows converging on a region = an attractor; the looping steer should deepen and narrow it, REBUS
re-open it, and the norm-matched `random` push should *not* produce a coherent basin. See
`context/basin_flow.md` for the method, citations (DMET 2505.20340; adversarial-states 2503.09066),
and the honest scope (a 2-D PCA window, exploratory companion — not the proof).

In [ ]:
# Experiment ③·b — basin landscape. Reuses the same vectors/conditions as Experiment ③.
import numpy as np
import matplotlib.pyplot as plt
from steering import basins
from steering.probes_eval import GEN_PROMPTS, build_conditions

# Inject at a decoder layer at the top of the steered band; read its output one layer on.
INJECT_LAYER = BAND[-1]                       # 0-based decoder-layer index (into the ModuleList)
SEED_PROMPT  = GEN_PROMPTS[0]                 # fixed context; we overwrite its last-token activation
GRID_N       = 12

# 1) Fit the PCA basis ONCE, on CLEAN activations, so every condition shares coordinates.
#    Collect the output of decoder INJECT_LAYER == hidden_states[INJECT_LAYER + 1], last token.
acts = basins.collect_residual(cmodel, tok, GEN_PROMPTS, layer=INJECT_LAYER + 1, token_pos=-1)
pca  = basins.PCA2.fit(acts)
Z    = pca.project(acts)
EXTENT = float(2.5 * Z.std())                 # grid half-width in PCA units, from the data spread
print(f"inject_layer={INJECT_LAYER}  explained={pca.explained}  extent={EXTENT:.2f}  n_acts={len(acts)}")

# 2) Same four arms as Experiment ③ (identical random vector via the shared seed).
conds = build_conditions(looping_vec, rebus_vec, coeff=COEFF, seed=0)

# 3) Flow field per condition on the shared grid + basis.
fields = {}
for label, vec, c in conds:
    fields[label] = basins.flow_field(
        cmodel, tok, pca, layer=INJECT_LAYER, seed_prompt=SEED_PROMPT,
        grid_n=GRID_N, extent=EXTENT, vector=vec, coeff=c)
    print(f"  {label:8s} mean flow speed = {fields[label]['speed'].mean():.3f}")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, label in zip(axes, ["clean", "looping", "REBUS", "random"]):
    basins.quiver_plot(fields[label], ax=ax,
                       title=f"{label}  (mean flow={fields[label]['speed'].mean():.2f})")
fig.suptitle("Residual-stream basin under each steer (arrows = one-layer flow)")
plt.tight_layout(); plt.show()

# 4) Dose-response: how far out does the basin still pull the state back? (width = 0.5 crossing)
profiles = {}
for label, vec, c in conds:
    profiles[label] = basins.basin_profile(
        cmodel, tok, pca, layer=INJECT_LAYER, seed_prompt=SEED_PROMPT,
        center_z=(0.0, 0.0), vector=vec, coeff=c)
basins.plot_basin_profile(profiles, title="basin return curve - inward fraction vs teleport radius")
plt.show()

# 5) Headline read: looping basin should be tighter (lower mean flow) and pull back farther than random.
print("\nbasin summary (mean flow speed; lower near center = tighter basin):")
for label in ["clean", "looping", "REBUS", "random"]:
    inw = profiles[label]["inward_fraction"]
    half = profiles[label]["radii"][np.argmax(inw < 0.5)] if (inw < 0.5).any() else float("inf")
    print(f"  {label:8s} mean_flow={fields[label]['speed'].mean():.3f}  basin_width(0.5 cross)~{half}")
